Cell 1: Setup & Initialization

In [0]:
# MAGIC %md
# MAGIC # ⚡ EV Site Selection — NB02: Silver Layer
# MAGIC **Layer:** Silver | **Objective:** รวม POI, Population, Flood Risk ลง Grid และแบ่งโซนพื้นที่

import pandas as pd
from pyspark.sql.functions import col, lit, count, when, round as spark_round

DATABASE_NAME = "ev_site_selection"
spark.sql(f"USE {DATABASE_NAME}")

GRID_LAT_SIZE = 0.0045
GRID_LON_SIZE = 0.0051

print(f"✅ Setup Complete | Database: {DATABASE_NAME}")

✅ Setup Complete | Database: ev_site_selection


Cell 2: Merge Base Grid (Population + Flood Risk)

In [0]:
# MAGIC %md
# MAGIC ## Cell 2: Merge Base Grid (Population + Flood Risk)

print("⏳ Merging Master Grid with Population and Flood Risk...")

df_master = spark.table("master_grid")
df_pop    = spark.table("bronze_population").select("grid_lat", "grid_lon", "pop_density")
df_flood  = spark.table("bronze_flood_risk").select("grid_lat", "grid_lon", "flood_occurrence")

df_silver_base = df_master \
    .join(df_pop,   on=["grid_lat", "grid_lon"], how="left") \
    .join(df_flood, on=["grid_lat", "grid_lon"], how="left") \
    .fillna({"pop_density": 0.0, "flood_occurrence": 0.0})

df_silver_base.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_grid_base")

print(f"✅ silver_grid_base: {df_silver_base.count():,} cells")
display(df_silver_base.orderBy(col("pop_density").desc()).limit(5))

⏳ Merging Master Grid with Population and Flood Risk...
✅ silver_grid_base: 18,492 cells


grid_lat,grid_lon,pop_density,flood_occurrence
13.774,100.6376,35230.53125,0.0
13.7695,100.6376,35230.53125,0.0
13.7605,100.6478,30033.28515625,0.0
13.7605,100.6427,30033.28515625,0.0
13.765,100.6427,30033.28515625,0.0


Cell 3: Map POI to Nearest Grid (Spatial Snapping)

In [0]:
# MAGIC %md
# MAGIC ## Cell 3: Map POI to Nearest Grid (Spatial Snapping)

from pyspark.sql.functions import round as spark_round, col

SOUTH = 13.45
WEST  = 100.25

print("⏳ Snapping POI coordinates to Master Grid...")

df_poi_raw = spark.table("bronze_osm_poi")

# สูตร: round((Point - Start) / Step) * Step + Start
df_poi_mapped = df_poi_raw \
    .withColumn("grid_lat",
        spark_round((col("lat") - SOUTH) / GRID_LAT_SIZE) * GRID_LAT_SIZE + SOUTH) \
    .withColumn("grid_lon",
        spark_round((col("lon") - WEST)  / GRID_LON_SIZE) * GRID_LON_SIZE + WEST)

df_poi_mapped = df_poi_mapped \
    .withColumn("grid_lat", spark_round("grid_lat", 4)) \
    .withColumn("grid_lon", spark_round("grid_lon", 4))

print("✅ POI snapped to grid")
display(df_poi_mapped.select("name", "poi_type", "lat", "grid_lat", "lon", "grid_lon").limit(5))

⏳ Snapping POI coordinates to Master Grid...
✅ POI snapped to grid


name,poi_type,lat,grid_lat,lon,grid_lon
Unknown,charging_station,13.9884205,13.99,100.6177918,100.6172
Unknown,charging_station,13.9207342,13.9225,100.6008293,100.6019
Unknown,charging_station,13.694362,13.693,100.5000419,100.4999
Unknown,charging_station,13.6952561,13.6975,100.6057795,100.607
Unknown,charging_station,13.7661754,13.765,100.6548647,100.6529


Cell 4: Aggregate POI Counts per Grid

In [0]:
# MAGIC %md
# MAGIC ## Cell 4: Aggregate POI Counts per Grid

print("⏳ Counting POIs per grid cell...")

df_poi_counts = df_poi_mapped \
    .groupBy("grid_lat", "grid_lon") \
    .pivot("poi_type") \
    .count() \
    .fillna(0)

for col_name in df_poi_counts.columns:
    if col_name not in ["grid_lat", "grid_lon"]:
        df_poi_counts = df_poi_counts.withColumnRenamed(col_name, f"cnt_{col_name}")

df_poi_counts.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DATABASE_NAME}.silver_poi_counts")

print(f"✅ silver_poi_counts: {df_poi_counts.count():,} grids with POI data")

⏳ Counting POIs per grid cell...
✅ silver_poi_counts: 1,236 grids with POI data


Cell 5: Feature Integration & Zoning Logic

In [0]:
# MAGIC %md
# MAGIC ## Cell 5: Feature Integration & Zoning Logic

from pyspark.sql.functions import col, when

print("⏳ Joining features and applying zoning logic...")

df_silver_joined = spark.table(f"{DATABASE_NAME}.silver_grid_base") \
    .join(spark.table(f"{DATABASE_NAME}.silver_poi_counts"),
          on=["grid_lat", "grid_lon"], how="left") \
    .fillna(0)

pop_threshold_high = df_silver_joined.approxQuantile("pop_density", [0.8], 0.05)[0]
pop_threshold_med  = df_silver_joined.approxQuantile("pop_density", [0.5], 0.05)[0]
columns_available  = df_silver_joined.columns

def get_poi_condition(col_name):
    """คืนค่า condition ของ column ถ้ามีอยู่ มิฉะนั้นคืน False"""
    return col(col_name) > 0 if col_name in columns_available else lit(False)

df_silver_zoned = df_silver_joined.withColumn("area_zone",
    when(
        (col("pop_density") >= pop_threshold_high) &
        (get_poi_condition("cnt_shopping_mall") | get_poi_condition("cnt_office_building")),
        "CBD"
    ).when(
        (col("pop_density") >= pop_threshold_med) | get_poi_condition("cnt_fuel_station"),
        "Urban"
    ).otherwise("Suburban")
)

print("✅ Zoning Logic Applied")
display(df_silver_zoned.groupBy("area_zone").count())


⏳ Joining features and applying zoning logic...
✅ Zoning Logic Applied


area_zone,count
CBD,359
Urban,9468
Suburban,8665


Cell 6: Preliminary Scoring & Save Silver Layer

In [0]:
# MAGIC %md
# MAGIC ## Cell 6: Preliminary Scoring & Save Silver Layer

print("⏳ Calculating preliminary potential score...")

def get_score_col(col_name, multiplier):
    """คืนค่า weighted column ถ้ามีอยู่ มิฉะนั้นคืน 0"""
    return (col(col_name) * multiplier) if col_name in df_silver_zoned.columns else lit(0.0)

df_silver_final = df_silver_zoned.withColumn("potential_score",
    (col("pop_density")                       * 0.4) +
    get_score_col("cnt_shopping_mall",          2.0) +
    get_score_col("cnt_office_building",        1.5) +
    get_score_col("cnt_hotel",                  1.0) -
    (col("flood_occurrence")                  * 1.0)
)

df_silver_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DATABASE_NAME}.silver_ev_features")

print(f"📦 silver_ev_features: {df_silver_final.count():,} records saved")
display(df_silver_final.orderBy(col("potential_score").desc()).limit(10))


⏳ Calculating preliminary potential score...
📦 silver_ev_features: 18,492 records saved


grid_lat,grid_lon,pop_density,flood_occurrence,cnt_charging_station,cnt_fuel_station,cnt_hotel,cnt_office_building,cnt_other,cnt_shopping_mall,area_zone,potential_score
13.7695,100.6376,35230.53125,0.0,0,2,0,0,0,0,Urban,14092.212500000001
13.774,100.6376,35230.53125,0.0,0,0,0,0,0,0,Urban,14092.212500000001
13.7605,100.6427,30033.28515625,0.0,0,0,2,1,0,0,CBD,12016.814062500001
13.7605,100.6478,30033.28515625,0.0,0,0,0,0,0,0,Urban,12013.314062500001
13.765,100.6478,30033.28515625,0.0,1,0,0,0,0,0,Urban,12013.314062500001
13.765,100.6427,30033.28515625,0.0,2,0,0,0,0,0,Urban,12013.314062500001
13.7695,100.6325,29875.89453125,0.0,0,0,4,0,0,0,Urban,11954.3578125
13.774,100.6325,29875.89453125,0.0,0,0,0,1,0,0,CBD,11951.8578125
13.7695,100.6274,29875.89453125,0.0,0,0,0,0,0,0,Urban,11950.3578125
13.774,100.6274,29875.89453125,0.0,0,0,0,0,0,0,Urban,11950.3578125


Cell 7: Silver Layer Quality Check

In [0]:
# MAGIC %md
# MAGIC ## Cell 7: Silver Layer Quality Check

from pyspark.sql.functions import col, count, when

all_tables = [
    "master_grid", "bronze_osm_poi", "bronze_population",
    "bronze_flood_risk", "silver_grid_base", "silver_poi_counts", "silver_ev_features",
]

print("📊 Pipeline Status Check\n")
print(f"{'Table':<25} | {'Records':<12} | Status")
print("-" * 55)

for t in all_tables:
    try:
        c      = spark.table(t).count()
        status = "✅ OK" if c > 0 else "⚠️ Empty"
        print(f"{t:<25} | {c:,<12} | {status}")
    except Exception:
        print(f"{t:<25} | {'0':<12} | ❌ Missing")

print("-" * 55)

df_final    = spark.table("silver_ev_features")
null_report = df_final.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["pop_density", "potential_score", "area_zone"]
])
print("\n🔍 Null counts in key columns:")
display(null_report)

print("\n🏙️ Zone distribution:")
display(df_final.groupBy("area_zone").count().orderBy("count", ascending=False))

print("\n💎 Top 5 locations by potential score:")
display(df_final.orderBy(col("potential_score").desc()).limit(5))

📊 Pipeline Status Check

Table                     | Records      | Status
-------------------------------------------------------
master_grid               | 18492,,,,,,, | ✅ OK
bronze_osm_poi            | 2453,,,,,,,, | ✅ OK
bronze_population         | 18492,,,,,,, | ✅ OK
bronze_flood_risk         | 18492,,,,,,, | ✅ OK
silver_grid_base          | 18492,,,,,,, | ✅ OK
silver_poi_counts         | 1236,,,,,,,, | ✅ OK
silver_ev_features        | 18492,,,,,,, | ✅ OK
-------------------------------------------------------

🔍 Null counts in key columns:


pop_density,potential_score,area_zone
0,0,0



🏙️ Zone distribution:


area_zone,count
Urban,9468
Suburban,8665
CBD,359



💎 Top 5 locations by potential score:


grid_lat,grid_lon,pop_density,flood_occurrence,cnt_charging_station,cnt_fuel_station,cnt_hotel,cnt_office_building,cnt_other,cnt_shopping_mall,area_zone,potential_score
13.774,100.6376,35230.53125,0.0,0,0,0,0,0,0,Urban,14092.212500000001
13.7695,100.6376,35230.53125,0.0,0,2,0,0,0,0,Urban,14092.212500000001
13.7605,100.6427,30033.28515625,0.0,0,0,2,1,0,0,CBD,12016.814062500001
13.7605,100.6478,30033.28515625,0.0,0,0,0,0,0,0,Urban,12013.314062500001
13.765,100.6478,30033.28515625,0.0,1,0,0,0,0,0,Urban,12013.314062500001
